In [2]:
import torch
from torch import nn
from torch.nn import functional as F
torch.manual_seed(42)

In [3]:
class VQVAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        # kernel = stride = 2 → 畳み込みの窓が重ならない
        # 2 層で縦横 1/4 になる（28 → 7）。潜在ベクトル 1 つ => 4×4 ピクセルのパッチ 1 つ
        self.enc = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=2, stride=2),
            nn.GELU(),
            nn.Conv2d(in_channels=32, out_channels = latent_dim, kernel_size=2, stride=2)
        )

        # エンコーダと対称な構成（7 → 28 に戻す）
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(in_channels = latent_dim, out_channels=32, kernel_size=2, stride=2),
            nn.GELU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=2, stride=2),
            nn.Sigmoid()
        )
        
x = torch.rand((4, 1, 28, 28))
print(x.shape)
vqvae = VQVAE(latent_dim=4)
z = vqvae.enc(x)
print(z.shape)
y = vqvae.dec(z)
print(y.shape)

torch.Size([4, 1, 28, 28])
torch.Size([4, 4, 7, 7])
torch.Size([4, 1, 28, 28])


In [4]:
class VectorQuantizer(nn.Module):
    def __init__(self, codebook_size: int, latent_dim: int):
        super().__init__()

        self.embbeding = nn.Embedding(codebook_size, latent_dim)
        nn.init.uniform_(self.embbeding.weight, 0, 1)

    def quantize(self, x):
        idx = torch.cdist(x, self.embbeding.weight).argmin(-1) # argminで勾配が消失しまう
        z = self.embbeding(idx)
        # z_q = x + (z - x).detach()    
        return z
    
    def forward(self, x):
        return self.quantize(x)

vq =VectorQuantizer(8, 2)
rand = torch.rand((1,2))

In [5]:
import re
from torchview import draw_graph

model_graph = draw_graph(vq, input_size=(1, 2))
graph = model_graph.visual_graph

for line in list(graph.body):
    if "label=" in line and re.search(r"\bargmin\b", line):
        match = re.match(r'\s*(".*?"|\S+)\s+\[', line)
        if match:
            graph.node(
                match.group(1).strip('"'),
                style="filled",
                fillcolor="pink",
            )
graph.render("non-ste", format="pdf", cleanup=True)


(process:37096): Pango-WARNING **: 15:42:18.515: couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.


'non-ste.pdf'

In [6]:
# DIM   : 潜在ベクトルの次元数
# CODES : コードブックのサイズ（コード数）
# BETA  : Commitment Loss の重み
DIM, CODES, BETA = 4, 64, 0.25

class VQVAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        # kernel = stride = 2 → 畳み込みの窓が重ならない
        # 2 層で縦横 1/4 になる（28 → 7）。潜在ベクトル 1 つ => 4×4 ピクセルのパッチ 1 つ
        self.enc = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=2, stride=2),
            nn.GELU(),
            nn.Conv2d(in_channels=32, out_channels = latent_dim, kernel_size=2, stride=2)
        )

        # エンコーダと対称な構成（7 → 28 に戻す）
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(in_channels = latent_dim, out_channels=32, kernel_size=2, stride=2),
            nn.GELU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=2, stride=2),
            nn.Sigmoid()
        )
        
        self.quantizer = VectorQuantizer(num_embeddings=CODES, embedding_dim=DIM)
        
    def encode(self, x):
        z_raw = self.enc(x)
        # チャネル（＝潜在次元）を最後に移し、位置ごとのベクトルをすべて一行に並べる
        z = z_raw.permute(0, 2, 3, 1).reshape(-1, DIM)
        z_q = self.quantizer(z)
        return z_q
    
    def decode(self, z_q):
        return self.decode(z_q)
    
    def forward(self, x):
        z_q = self.encode(x)
        return self.decode(z_q)   

In [7]:
# DIM   : 潜在ベクトルの次元数
# CODES : コードブックのサイズ（コード数）
# BETA  : Commitment Loss の重み
DIM, CODES, BETA = 4, 64, 0.25

class StraightThrough(nn.Module):
    def forward(self, x, z):
        return x + (z - x).detach()

class VectorQuantizer(nn.Module):
    def __init__(self, codebook_size: int, latent_dim: int):
        super().__init__()

        self.embbeding = nn.Embedding(codebook_size, latent_dim)
        nn.init.uniform_(self.embbeding.weight, 0, 1)
        
        self.ste = StraightThrough()

    def quantize(self, x):
        idx = torch.cdist(x, self.embbeding.weight).argmin(-1) # argminで勾配が消失しまう
        z = self.embbeding(idx)
        # return z # Straight-Through　不使用
        z_q = self.ste(x, z)
        return z_q
    
    def forward(self, x):
        return self.quantize(x)

class VQVAE(nn.Module):
    def __init__(self):
        super().__init__()
        # kernel = stride = 2 → 畳み込みの窓が重ならない
        # 2 層で縦横 1/4 になる（28 → 7）。潜在ベクトル 1 つ => 4×4 ピクセルのパッチ 1 つ
        self.enc = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=2, stride=2),
            nn.GELU(),
            nn.Conv2d(in_channels=32, out_channels=DIM, kernel_size=2, stride=2)
        )

        # エンコーダと対称な構成（7 → 28 に戻す）
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(in_channels=DIM, out_channels=32, kernel_size=2, stride=2),
            nn.GELU(),
            nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=2, stride=2),
            nn.Sigmoid()
        )

        # コードブック：CODES 個の DIM 次元ベクトル
        self.codebook = nn.Embedding(CODES, DIM)
        nn.init.uniform_(self.codebook.weight, -1 / CODES, 1 / CODES)

    def encode(self, x):
        """
        x: (B, 1, 28, 28)
        戻り値: z   (B*7*7, DIM) 潜在ベクトル
                idx (B*7*7,)     各ベクトルに最も近いコード番号
        """
        z_raw = self.enc(x)  # (B, DIM, 7, 7)

        # チャネル（＝潜在次元）を最後に移し、位置ごとのベクトルをすべて一行に並べる
        z = z_raw.permute(0, 2, 3, 1).reshape(-1, DIM)
        
        # 全コードとのユークリッド距離を計算し、最も近いコードを選ぶ
        return z, torch.cdist(z, self.codebook.weight).argmin(1)

    def forward(self, x):
        z, idx = self.encode(x)
        q = self.codebook(idx)  # 量子化後のベクトル

        # 第1項: コードブックをエンコーダ出力に近づける（エンコーダ側は固定）
        # 第2項: エンコーダ出力をコードに近づける＝Commitment Loss（コードブック側は固定）
        codebook_loss = F.mse_loss(q, z.detach())
        commitment_loss = F.mse_loss(z, q.detach())
        vq_loss = codebook_loss + BETA * commitment_loss

        # Straight-Through 推定：順伝播では q を使い、逆伝播では勾配を z にそのまま流す（argmin は微分できないため）
        q = z + (q - z).detach()

        # (B*7*7, DIM) → (B, DIM, 7, 7) に戻してデコーダへ
        q = q.view(len(x), 7, 7, DIM).permute(0, 3, 1, 2)
        return self.dec(q), vq_loss

In [12]:

vqvae =VQVAE()
rand = torch.rand((28, 28))

model_graph = draw_graph(vqvae, input_size=(1, 1, 28, 28))
graph = model_graph.visual_graph

for line in list(graph.body):
    if "label=" in line and re.search(r"\bargmin\b", line):
        match = re.match(r'\s*(".*?"|\S+)\s+\[', line)
        if match:
            graph.node(
                match.group(1).strip('"'),
                style="filled",
                fillcolor="pink",
            )
graph.render("vqvae", format="pdf", cleanup=True)


(process:30864): Pango-WARNING **: 15:43:32.631: couldn't load font "Linux libertine Not-Rotated 10", falling back to "Sans Not-Rotated 10", expect ugly output.


'vqvae.pdf'

In [9]:
class ResidualVectorQuantizer(nn.Module):
    def __init__(self, num_quantizers: int, codebook_size: int, latent_dim: int):
        super().__init__()
        self.layers = nn.ModuleList(
            [VectorQuantizer(codebook_size, latent_dim) for _ in range(num_quantizers)]
        )
 
    def quantize(self, x):
        residual, q_sum = x, torch.zeros_like(x)
        for vq in self.layers:
            
            q = vq.quantize(residual)     # 各段は普通の VQ
            q_sum = q_sum + q.detach()
            residual = residual - q.detach()   # 表せなかった残りを次の段へ

        return q_sum
    
    def forward(self, x):
        return self.quantize(x)
    
rvq =ResidualVectorQuantizer(2, 8, 2)
rand = torch.rand((1,2))
model_graph = model_graph = draw_graph(
    rvq,
    input_size=(1, 2),
    depth=1,
    show_shapes=False,
    hide_inner_tensors=True,
    hide_module_functions=True,
    expand_nested=False,
)
graph = model_graph.visual_graph

for line in list(graph.body):
    if "label=" in line and re.search(r"\bargmin\b", line):
        match = re.match(r'\s*(".*?"|\S+)\s+\[', line)
        if match:
            graph.node(
                match.group(1).strip('"'),
                style="filled",
                fillcolor="pink",
            )
# graph.render("non-ste", format="pdf", cleanup=True)
graph.save("./model.gv")

'.\\./model.gv'